Paysafe AI solution

In [65]:
from pydantic import BaseModel, Field, model_validator
from typing import List, Optional
from enum import Enum

class PrimaryIntent(str, Enum):
    account_access = "account_access"
    account_creation = "account_creation"
    verification = "verification"
    payments_and_funds = "payments_and_funds"
    general_enquiry = "general_enquiry"

class RiskLevel(str, Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"

class SubIntent(str, Enum):
    reset_pin = "reset_pin"
    reset_secure_id = "reset_secure_id"
    sms_not_received = "sms_not_received"
    account_restricted = "account_restricted"
    close_account = "close_account"
    create_new_account = "create_new_account"
    restore_old_account = "restore_old_account"
    address_verification = "address_verification"
    identity_verification = "identity_verification"
    phone_number_verification = "phone_number_verification"
    payment_declined = "payment_declined"
    withdrawal_help = "withdrawal_help"
    NULL = "NULL"


PRIMARY_TO_SUB = {
    PrimaryIntent.account_access: {
        SubIntent.reset_pin,
        SubIntent.reset_secure_id,
        SubIntent.sms_not_received,
        SubIntent.account_restricted,
        SubIntent.close_account,
    },
    PrimaryIntent.account_creation: {
        SubIntent.create_new_account,
        SubIntent.restore_old_account,
    },
    PrimaryIntent.verification: {
        SubIntent.address_verification,
        SubIntent.identity_verification,
        SubIntent.phone_number_verification,
    },
    PrimaryIntent.payments_and_funds: {
        SubIntent.payment_declined,
        SubIntent.withdrawal_help,
    },
    PrimaryIntent.general_enquiry: {
        SubIntent.NULL,
    },
}

class SupportAnalysis(BaseModel):
    primary_intent: PrimaryIntent
    primary_sub_intent: Optional[SubIntent]

    secondary_intent: Optional[PrimaryIntent]
    secondary_sub_intent: Optional[SubIntent]

    key_information: List[str]
    risk_level: RiskLevel
    escalation_required: bool
    confidence_score: float = Field(ge=0.0, le=1.0)
    manual_review_required: bool
    suggested_next_steps: List[str]

    @model_validator(mode="after")
    def validate_intent_hierarchy(self):

        if self.primary_sub_intent and self.primary_sub_intent != SubIntent.NULL:
            valid_subs = PRIMARY_TO_SUB.get(self.primary_intent, set())
            if self.primary_sub_intent not in valid_subs:
                raise ValueError(
                    f"{self.primary_sub_intent} does not belong to {self.primary_intent}"
                )
            
        if self.secondary_sub_intent and not self.secondary_intent:
            raise ValueError("secondary_sub_intent provided without secondary_intent")

        if self.secondary_intent and self.secondary_sub_intent:
            valid_subs = PRIMARY_TO_SUB.get(self.secondary_intent, set())
            if self.secondary_sub_intent not in valid_subs:
                raise ValueError(
                    f"{self.secondary_sub_intent} does not belong to {self.secondary_intent}"
                )

        return self

In [66]:
import pandas as pd

df = pd.read_excel('Customer_Contact_Emails_20.xlsx')
print(df.head())

   ID                                               TEXT
0   1  I'm getting a message that says to contact you...
1   2  I have entered correct phone number to verify ...
2   3  I receive message saying I can not pay merchan...
3   4  My camera is damaged and I can't verify my inf...
4   5  Account verification is not allowing my mobile...


In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv
import json
from pydantic import ValidationError

load_dotenv()

client = OpenAI(api_key=os.getenv("API_KEY"))



def analyse_query(query: str) -> SupportAnalysis:
    system_prompt = open('system_prompt.txt', 'r').read()
    response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f'Customer Enquiry:\n"{query}"\n\nReturn the analysis in the specified JSON format.'}
    ]
)

    analysis = response.choices[0].message.content
    return analysis


def parse_response(response: str) -> SupportAnalysis:
    try:
        data = json.loads(response)
    except json.JSONDecodeError as e:
        raise ValueError("Invalid JSON response from model") from e

    try:
        return SupportAnalysis(**data)
    except ValidationError as e:
        raise ValueError("Model output failed schema validation") from e

In [68]:
def apply_risk_overrides(query: str, analysis: SupportAnalysis) -> SupportAnalysis:
    
    high_risk_indicators = ["emergency", "complaint", "legal threat", "fraud", "identity theft", "ombudsman", "regulator", "escalate", "supervisor", "manager"]
    if any(high_risk_indicator in query.lower() for high_risk_indicator in high_risk_indicators):
        analysis.risk_level = RiskLevel.HIGH
        analysis.escalation_required = True
    return analysis



In [ ]:
def adjust_confidence(analysis: SupportAnalysis) -> SupportAnalysis:
    confidence = analysis.confidence_score

    if analysis.secondary_intent is not None:
        confidence -= 0.1


    if (
        analysis.secondary_sub_intent is not None
        and analysis.secondary_sub_intent != SubIntent.NULL
    ):
        confidence -= 0.1


    confidence = max(0.0, min(1.0, confidence))

    analysis.confidence_score = confidence
    analysis.manual_review_required = confidence < 0.6

    return analysis

In [70]:
sample_text = df.iloc[0]['TEXT']
raw_analysis = analyse_query(sample_text)
print("Raw Analysis:", raw_analysis)
try:
    structured_analysis = parse_response(raw_analysis)
    structured_analysis = apply_risk_overrides(sample_text, structured_analysis)
    structured_analysis = adjust_confidence(structured_analysis)
    print("structured analysis datatype:", type(structured_analysis))
    print("Final Analysis after Risk Overrides and Confidence Adjustment:", structured_analysis)
except ValueError as e:
    print("Error parsing response:", e)

Raw Analysis: {
  "primary_intent": "account_access",
  "primary_sub_intent": null,
  "secondary_intent": "general_enquiry",
  "secondary_sub_intent": null,
  "key_information": [
    "Customer reports receiving a message telling them to contact support about their account.",
    "Customer wants to use Skrill as a payment method."
  ],
  "risk_level": "LOW",
  "escalation_required": "false",
  "confidence_score": 0.65,
  "manual_review_required": "false",
  "reasoning_summary": "Primary intent is account_access due to account-related message; secondary intent is general_enquiry due to Skrill inquiry.",
  "suggested_next_steps": [
    "Request the exact message text the customer saw.",
    "Confirm whether Skrill is supported as a payment method.",
    "If supported, guide how to enable or link Skrill."
  ]
}
structured analysis datatype: <class '__main__.SupportAnalysis'>
Final Analysis after Risk Overrides and Confidence Adjustment: primary_intent=<PrimaryIntent.account_access: 'accou

In [71]:
results = []
for index, row in df.iterrows():
    query = row['TEXT']
    raw_analysis = analyse_query(query)
    try:
        structured_analysis = parse_response(raw_analysis)
        structured_analysis = apply_risk_overrides(query, structured_analysis)
        structured_analysis = adjust_confidence(structured_analysis)
        results.append(structured_analysis.model_dump())
    except ValueError as e:
        print(f"Error parsing response for query at index {index}: {e}")
results_df = pd.DataFrame(results)
results_df.to_csv('analysis_results.csv', index=False)

Error parsing response for query at index 3: 1 validation error for SupportAnalysis
  Value error, secondary_sub_intent provided without secondary_intent [type=value_error, input_value={'primary_intent': 'verif...r exception handling.']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
